# 02 | Pandas for Sales and Profit Analysis

**Amity University, Lucknow | MBA workshop | 9 October 2026**

![Python](https://img.shields.io/badge/Python-3-3776AB?logo=python&logoColor=white)
![NumPy](https://img.shields.io/badge/NumPy-Arrays-013243?logo=numpy&logoColor=white)
![Pandas](https://img.shields.io/badge/Pandas-Analytics-150458?logo=pandas&logoColor=white)
![Audience](https://img.shields.io/badge/Audience-MBA_students-00897B)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zaid-kamil/Data-analytics-using-numpy-and-python-at-Amity-Lucknow/blob/main/notebooks/02_pandas_sales_analysis.ipynb)

**Core session: 40 minutes + 10-minute exercise.** Move from raw transactions to a management summary.

### Your instructor: Zaid Kamil



## How to use this notebook
Run cells from top to bottom with **Shift + Enter**. Predict the result before running each example. All data is fictional and included in the notebook; no downloads or accounts beyond your notebook environment are needed.


## Learning outcomes
Load and inspect a table, identify quality issues, create business metrics, and compare performance by region and product.


## 1. Load a small CSV
The CSV is embedded so the example also works in Colab. `pd.read_csv("sales.csv")` follows the same pattern for a local file. Each row is a transaction; price and cost are **per unit**, in INR.


In [ ]:
import pandas as pd
from io import StringIO

csv_data = """Order_ID,Date,Product,Region,Units,Unit_Price,Unit_Cost
1001,2026-09-01,Notebook,Lucknow,20,100,60
1002,2026-09-02,Pen,Delhi,50,20,10
1003,2026-09-03,Notebook,Kanpur,15,100,60
1004,2026-09-04,Planner,Lucknow,10,300,180
1005,2026-09-05,Planner,Delhi,8,300,180
1006,2026-09-06,Pen,Kanpur,40,20,10
1007,2026-09-07,Notebook,Delhi,30,100,60
1008,2026-09-08,Planner,Lucknow,,300,189
1009,2026-09-09,Pen,Lucknow,60,20,10
1010,2026-09-10,Planner,Kanpur,5,300,180
1003,2026-09-03,Notebook,Kanpur,15,100,60
"""
raw = pd.read_csv(StringIO(csv_data))
raw.head()


## 2. Inspect before calculating
A DataFrame is a labelled table; a Series is a labelled one-dimensional collection, such as one column.


### Sample DataFrame: a sales table

Think of a DataFrame as a smart spreadsheet inside Python. Each row is one record, each column is one field, and the index labels the rows. Selecting one column gives a Pandas `Series`.

![Sample Pandas DataFrame](https://raw.githubusercontent.com/zaid-kamil/Data-analytics-using-numpy-and-python-at-Amity-Lucknow/main/assets/diagrams/pandas-dataframe.svg)


In [ ]:
print('Rows and columns:', raw.shape)
raw.info()




In [ ]:
print('\nMissing values:')
print(raw.isna().sum())

In [ ]:
print('\nRepeated order IDs:', raw['Order_ID'].duplicated().sum())

In [ ]:
raw.describe()

## 3. Clean with a stated business rule
This example assumes each Order_ID identifies one transaction. Keep the first repeated ID. Quarantine the row with missing Units rather than inventing sales; confirm it with the source before including it. Real orders with multiple line items need a different deduplication key.


In [ ]:
deduplicated = raw.drop_duplicates(subset=['Order_ID']).copy()
review_needed = deduplicated.loc[deduplicated['Units'].isna()].copy()

sales = deduplicated.dropna(subset=['Units']).copy()
sales['Date'] = pd.to_datetime(sales['Date'])
sales['Units'] = sales['Units'].astype(int)
print('Usable transactions:', len(sales))
print('Transactions needing review:', len(review_needed))
review_needed


## 4. Create revenue and profit columns
Revenue = Units × Unit Price. Gross profit = Revenue − product cost. This excludes overheads, taxes, and other operating expenses.


In [ ]:
sales['Revenue'] = sales['Units'] * sales['Unit_Price']
sales['Total_Cost'] = sales['Units'] * sales['Unit_Cost']
sales['Gross_Profit'] = sales['Revenue'] - sales['Total_Cost']
sales['Margin_Pct'] = (100 * sales['Gross_Profit'] / sales['Revenue'].replace(0, float('nan'))).round(2)
sales[['Order_ID', 'Product', 'Revenue', 'Gross_Profit', 'Margin_Pct']]


## 5. Select and filter
**Example A:** view Lucknow transactions.

**Example B:** find transactions with revenue of at least INR 2,000.


In [ ]:
print(sales.loc[sales['Region'] == 'Lucknow', ['Order_ID', 'Product', 'Revenue']])

In [ ]:
sales.loc[sales['Revenue'] >= 2000, ['Order_ID', 'Region', 'Revenue']]

## 6. Group into a management summary
Calculate regional totals and order counts. Regional margin must use total profit divided by total revenue, rather than an unweighted average of transaction percentages.


In [ ]:
region_summary = sales.groupby('Region').agg(
    Revenue=('Revenue', 'sum'),
    Gross_Profit=('Gross_Profit', 'sum'),
    Orders=('Order_ID', 'count')
)
region_summary['Margin_Pct'] = (100 * region_summary['Gross_Profit'] / region_summary['Revenue']).round(2)
region_summary.sort_values('Revenue', ascending=False)


## 7. Visualise the regional summary

A grouped bar chart lets us compare revenue and gross profit without interrogating every cell in the table.

![Revenue and gross profit by region](../assets/diagrams/pandas-region-chart.svg)

Pandas can send a DataFrame directly to Matplotlib with `.plot()`.


In [ ]:
import matplotlib.pyplot as plt

chart_data = region_summary[['Revenue', 'Gross_Profit']].sort_values('Revenue', ascending=False)
chart_data.plot(kind='bar', figsize=(8, 4), color=['#2563eb', '#22c55e'])
plt.title('Revenue and Gross Profit by Region')
plt.ylabel('INR')
plt.xlabel('Region')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 8. Rank products
Ranking by revenue and ranking by profit answer different business questions.


In [ ]:
sales

In [ ]:
product_summary = sales.groupby('Product')[['Revenue', 'Gross_Profit']].sum()
print('By revenue:')
print(product_summary.sort_values('Revenue', ascending=False))



In [ ]:
print('\nBy profit:')
print(product_summary.sort_values('Gross_Profit', ascending=False))

## 9. Optional: pivot and export
A pivot table provides an Excel-style cross-tab. The export stays in your notebook environment; in Colab, use the Files panel to download it.


In [ ]:
pivot = sales.pivot_table(index='Region', columns='Product', 
                          values='Revenue', aggfunc='sum', fill_value=0)
pivot


In [ ]:
region_summary.to_csv('regional_sales_summary.csv')

## 10. MBA mini-case
1. Which region leads in revenue?
2. Which product leads in gross profit?
3. Does the leading revenue region also lead in profit?
4. Why should the incomplete order be mentioned in your report?

Write a **two-sentence management recommendation**, separating facts from assumptions.


In [ ]:
# Write your analysis here.


### Suggested answer


In [ ]:
print('Highest revenue region:', region_summary['Revenue'].idxmax())
print('Highest profit product:', product_summary['Gross_Profit'].idxmax())
print('Highest profit region:', region_summary['Gross_Profit'].idxmax())
print('Revenue included:', sales['Revenue'].sum())
print('Excluded incomplete orders:', len(review_needed))


**Interpretation:** Delhi leads both recorded revenue and gross profit, while Planner leads product profit. Resolve the missing Lucknow order before treating this as a final performance report. Product mix can affect profitability, so revenue alone is insufficient.

**Next:** the third notebook introduces developer tools that support trustworthy analysis.


### Stay connected

If this session helped you, **follow [Zaid Kamil on GitHub](https://github.com/zaid-kamil)** for more learning resources and **[connect on LinkedIn](https://linkedin.com/in/zaid-kamil-94211a40)**. Mention the Amity Lucknow workshop in your connection request.

[![Follow on GitHub](https://img.shields.io/badge/GitHub-Follow_Zaid-181717?logo=github&logoColor=white)](https://github.com/zaid-kamil)
[![Connect on LinkedIn](https://img.shields.io/badge/LinkedIn-Connect-0A66C2)](https://linkedin.com/in/zaid-kamil-94211a40)

You can also star this repository to find the notebooks again.
